# Superstore Sales Data Cleaning & Feature Engineering

This notebook loads the raw Superstore dataset, investigates and resolves real data quality issues, engineers new features for downstream SQL/Power BI analysis, and exports a cleaned, pipe-delimited CSV.

## Step 1: Setup & Raw File Inspection

In [ ]:
import pandas as pd

Before loading the file into pandas,looking at the raw text directly. This is used later to confirm whether Postal Code values lose their leading zeros in the source file itself, or only during the pandas read.

In [ ]:
with open(r'D:\PL-300\Files\Superstore\Superstore.csv', encoding='latin1') as f:
    for i, line in enumerate(f):
        if i < 3:
            print(line)
        else:
            break

## Step 2: Load & Explore the Raw Data

In [ ]:
#Loading the Superstore dataset
# 'latin1' encoding needed because the file contains non-UTF8 characters
raw_sales_df = pd.read_csv(
    r'D:\PL-300\Files\Superstore\Superstore.csv',
    encoding='latin1',
    dtype={'Postal Code': str})

In [ ]:
#Checking first few rows to confirm if data loaded correctly or not
raw_sales_df.head()

In [ ]:
raw_sales_df.info()

In [ ]:
# Summary statistics for numeric columns (Sales, Quantity, Discount, Profit)
raw_sales_df.describe()

In [ ]:
# Check for missing values per column
raw_sales_df.isnull().sum()

In [ ]:
#Check for fully duplicated rows
raw_sales_df.duplicated().sum()

In [ ]:
# Spot-check categorical columns for inconsistent labeling (typos, casing, etc.)
raw_sales_df['Region'].unique()

In [ ]:
raw_sales_df['Ship Mode'].unique()

In [ ]:
raw_sales_df['Category'].unique()

## Step 3: Fix Data Types

In [ ]:
#Create cleaned_sales_df as a copy of raw_sales_df
cleaned_sales_df = raw_sales_df.copy()

In [ ]:
#Checking the raw date format
raw_sales_df['Order Date'][0]

In [ ]:
#Converting the order date and ship date to datetime64 dtype
cleaned_sales_df['Order Date'] = pd.to_datetime(cleaned_sales_df['Order Date'])

In [ ]:
cleaned_sales_df['Ship Date'] = pd.to_datetime(cleaned_sales_df['Ship Date'])

In [ ]:
cleaned_sales_df.info()

### Postal Code Investigation

`Postal Code` was read in as text via `dtype=str`, but a length check below reveals some values are only 4 digits instead of 5 — a sign leading zeros were lost somewhere upstream.

In [ ]:
cleaned_sales_df['Postal Code'].str.len().value_counts()

In [ ]:
# Look at which states are affected
cleaned_sales_df[cleaned_sales_df['Postal Code'].str.len() == 4]['State'].unique()

The raw source file itself has no leading zeros (confirmed via the raw text read in Step 1) — this is a genuine source-data issue, not a pandas parsing bug. The affected states are all known 0-prefix ZIP code regions, so the fix below is justified and documented.

In [ ]:
# NOTE: Postal Code arrived as int64 in source data, stripping leading zeros
# for 449 rows across 7 Northeastern states (CT, NJ, MA, RI, NH, VT, ME) —
# all known 0-prefix zip regions. Restored via zfill(5) since state
# matched known 0-prefix regions, but not verified against actual USPS records.
cleaned_sales_df['Postal Code'] = cleaned_sales_df['Postal Code'].str.zfill(5)
cleaned_sales_df['Postal Code'].str.len().value_counts()

In [ ]:
cleaned_sales_df[cleaned_sales_df['State'] == 'Massachusetts']['Postal Code'].unique()

### Date Sanity Checks

In [ ]:
#Checking the min and max of both date columns
print(cleaned_sales_df['Order Date'].min(), cleaned_sales_df['Order Date'].max())
print(cleaned_sales_df['Ship Date'].min(), cleaned_sales_df['Ship Date'].max())

In [ ]:
#Checking that Ship Date is never before Order Date
(cleaned_sales_df['Ship Date'] < cleaned_sales_df['Order Date']).sum()

In [ ]:
#Spot-check one row manually — compare the raw string from raw_sales_df to the converted value
# in cleaned_sales_df for the same row, and make sure day/month landed correctly
print(raw_sales_df['Order Date'][0], '→', cleaned_sales_df['Order Date'][0])

## Step 4: Feature Engineering

In [ ]:
#Order-to-Ship Days (days between order and shipment)
cleaned_sales_df['Order to Ship Days'] = (cleaned_sales_df['Ship Date'] - cleaned_sales_df['Order Date']).dt.days

In [ ]:
cleaned_sales_df['Order Year'] = cleaned_sales_df['Order Date'].dt.year
cleaned_sales_df['Order Month'] = cleaned_sales_df['Order Date'].dt.month
cleaned_sales_df['Order Quarter'] = cleaned_sales_df['Order Date'].dt.quarter

In [ ]:
cleaned_sales_df['Profit Margin'] = cleaned_sales_df['Profit'] / cleaned_sales_df['Sales'] * 100

Sanity check: confirm `Sales` never equals 0 before relying on it as a division denominator above.

In [ ]:
(cleaned_sales_df['Sales'] == 0).sum()

In [ ]:
cleaned_sales_df[['Order to Ship Days', 'Order Year', 'Order Month', 'Order Quarter', 'Profit Margin']].head()

In [ ]:
cleaned_sales_df['Order to Ship Days'].dtype

## Step 5: Standardize Column Names to PascalCase

Renaming at the Python stage (rather than later in SQL) keeps column names clean and consistent from the very start of the pipeline.

In [ ]:
cleaned_sales_df = cleaned_sales_df.rename(columns={
    'Row ID': 'RowID',
    'Order ID': 'OrderID',
    'Order Date': 'OrderDate',
    'Ship Date': 'ShipDate',
    'Ship Mode': 'ShipMode',
    'Customer ID': 'CustomerID',
    'Customer Name': 'CustomerName',
    'Postal Code': 'PostalCode',
    'Product ID': 'ProductID',
    'Sub-Category': 'SubCategory',
    'Product Name': 'ProductName',
    'Order to Ship Days': 'OrderToShipDays',
    'Order Year': 'OrderYear',
    'Order Month': 'OrderMonth',
    'Order Quarter': 'OrderQuarter',
    'Profit Margin': 'ProfitMargin'
})

## Step 6: Round Floating-Point Columns

Rounds monetary/percentage columns to 2 decimal places, removing floating-point precision artifacts (e.g., `16.000000000000004`) that would otherwise show up in the exported CSV.

In [ ]:
cleaned_sales_df['ProfitMargin'] = cleaned_sales_df['ProfitMargin'].round(2)
cleaned_sales_df['Sales'] = cleaned_sales_df['Sales'].round(2)
cleaned_sales_df['Profit'] = cleaned_sales_df['Profit'].round(2)
cleaned_sales_df['Discount'] = cleaned_sales_df['Discount'].round(2)

## Step 7: Export Cleaned Data

Exported as pipe-delimited (`|`) rather than comma-delimited, since some `Product Name` values contain embedded commas that would otherwise break downstream parsing (e.g., SQL Server's `BULK INSERT`).

In [ ]:
cleaned_sales_df.to_csv(r'D:\PL-300\Files\Superstore\cleaned_superstore.csv', index=False, sep='|')

Final verification: read the exported file back as raw text to confirm formatting (pipe delimiters, rounded decimals, intact Product Names) before handing off to SQL Server.

In [ ]:
with open(r'D:\PL-300\Files\Superstore\cleaned_superstore.csv', encoding='utf-8') as f:
    lines = f.readlines()
    for i in [2, 6, 15]:
        print(f"Row {i}: {lines[i]}")